# 04 - Architectural Patterns

Jev is designed to sit *inside* a larger system. The skill to practice is thinking in
discrete, atomic decisions that compose into system behavior. TypeSafe documents four
named patterns:

| Pattern | What it does | Buys you |
| --- | --- | --- |
| **Speculative fan-out** | Many questions in one call, including ones you might not need | Cost, speed |
| **Confidence-gated routing** | Confidence as a second axis: the answer says what, confidence says whether to act | Reliability, safety |
| **Composite scoring** | Split a complex judgment into atomic scores, weight them in code | Control, tunability |
| **Intent routing** | Classify first with a cheap call, invoke expensive handlers selectively | Cost, speed |

One principle powers all four: **questions in a request are evaluated in parallel and
independently.** Adding a question barely changes latency and costs only its own tokens.
One question's answer is never hidden context for another, so you can add or remove
questions without changing the others' results.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Find the TypeSafe API key: environment first, then any .env file up the tree.
if "TYPESAFE_API_KEY" not in os.environ:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / ".env").exists():
            load_dotenv(folder / ".env")
            break
assert os.environ.get("TYPESAFE_API_KEY"), "Set TYPESAFE_API_KEY (get a key at console.typesafe.ai/keys)"

import typesafe_sdk

print("typesafe-sdk", typesafe_sdk.__version__, "ready")

typesafe-sdk 0.7.5 ready


In [2]:
PRICE_PER_MTOK = 0.042  # dollars per million input tokens; output tokens are free


def cost_usd(response):
    """What this call cost, in dollars."""
    return response.usage.input_tokens / 1_000_000 * PRICE_PER_MTOK


def show(response):
    """Pretty-print every answer in a response, plus tokens and cost."""
    print(f"model={response.model}  input_tokens={response.usage.input_tokens}  cost=${cost_usd(response):.6f}")
    for qid, a in response.answers.items():
        if a.type == "noul":
            print(f"  {qid:<28} noul={a.noul:.2f}")
        elif a.type == "choice":
            print(f"  {qid:<28} choice={a.choice!r}  confidence={a.confidence:.2f}")
            print(f"  {'':<28} probabilities={a.probabilities}")
        elif a.type == "score":
            print(f"  {qid:<28} score={a.score:.2f}  confidence={a.confidence:.2f}")
            print(f"  {'':<28} probabilities={a.probabilities}")
        else:
            print(f"  {qid:<28} {a}")

## Pattern 1: Speculative fan-out

Ask every question your code *might* need, and let code decide what is relevant afterwards.
Here `bug_severity` and `has_repro_steps` only matter if the ticket is a bug report, and
`refund_requested` only matters for billing. We ask all of them anyway, in one call.

In [3]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

TRIAGE = {
    "category": Choice(
        instructions="What kind of ticket is this?",
        criteria={
            "bug_report": "Something is broken or not working as documented",
            "billing": "Charges, invoices, refunds, subscriptions",
            "feature_request": "Asking for new functionality",
        },
    ),
    "bug_severity": Score(
        instructions="If this reports a bug, how severe is it?",
        criteria=[
            "Cosmetic; no impact to functionality",
            "Broken or degraded feature, but workaround exists",
            "Blocking issue; no workaround exists",
        ],
    ),
    "has_repro_steps": Noul(instructions="Does the ticket include steps to reproduce the problem?"),
    "refund_requested": Noul(instructions="Does the customer explicitly request a refund?"),
    "frustration": Score(
        instructions="How frustrated does the customer appear?",
        criteria=["Calm, just stating facts", "Frustrated but civil", "Very angry, strong language"],
    ),
}

ticket = (
    "Since yesterday the webhook retries fire twice for every event. "
    "Steps: create any order, watch the webhook log, each delivery appears twice. "
    "This is corrupting our inventory counts and I am honestly losing patience."
)

with TypeSafeClient() as client:
    response = client.system_one(state=ticket, questions=TRIAGE)

show(response)

model=jev-1.13.0  input_tokens=525  cost=$0.000022
  category                     choice='bug_report'  confidence=1.00
                               probabilities={'bug_report': 1.0, 'billing': 0.0, 'feature_request': 0.0}
  bug_severity                 score=1.48  confidence=0.28
                               probabilities={0: 0.0, 1: 0.52, 2: 0.48}
  has_repro_steps              noul=0.96
  refund_requested             noul=0.02
  frustration                  score=1.06  confidence=0.91
                               probabilities={0: 0.0, 1: 0.94, 2: 0.06}


In [4]:
answers = response.answers

if answers["category"].choice == "bug_report":
    if answers["bug_severity"].score > 1.5 and answers["has_repro_steps"].noul > 0.6:
        print("-> escalate_to_engineering(severity='high')")
    else:
        print("-> add_to_bug_backlog()")
elif answers["category"].choice == "billing":
    print("-> route_to_billing(refund_likely =", answers["refund_requested"].noul > 0.7, ")")
else:
    print("-> log_feature_request()")

# frustration is useful regardless of category
if answers["frustration"].score > 1.5:
    print("-> flag_for_priority_response()")

# refund_requested was speculative for this ticket; the code simply never read it. That is the pattern.

-> add_to_bug_backlog()


## The economics of batching, measured

The docs claim batching 13 questions into one call is about 11.5x cheaper and 9.6x faster
than 13 separate calls, with no change in the answers. Why: every call pays to send the
state; N calls pay for the state N times, plus N round trips.

Let's verify the shape of that claim ourselves with an 8-question compliance checklist over
one policy paragraph.

In [5]:
import time

from typesafe_sdk import Noul, TypeSafeClient

policy = (
    "We retain personal data for 90 days after account closure, after which it is "
    "deleted from production systems. Backups age out within 12 months. Users may "
    "request a copy of their data at any time by contacting support. We share data "
    "with payment processors solely to complete transactions, and we notify users "
    "of material changes to this policy by email at least 30 days in advance."
)

CHECKLIST = {
    "retention_defined": Noul(instructions="Does the policy state a retention period for personal data?"),
    "backup_lifecycle": Noul(instructions="Does the policy say how long backups are kept?"),
    "data_export": Noul(instructions="Can users request a copy of their data?"),
    "third_party_sharing": Noul(instructions="Does the policy disclose sharing data with third parties?"),
    "sharing_purpose_limited": Noul(instructions="Is third-party sharing limited to a stated purpose?"),
    "change_notification": Noul(instructions="Are users notified before policy changes take effect?"),
    "deletion_on_closure": Noul(instructions="Is data deleted after account closure?"),
    "mentions_encryption": Noul(instructions="Does the policy mention encryption?"),
}

with TypeSafeClient() as client:
    # one question per call, sequentially
    t0 = time.perf_counter()
    single_tokens, single_answers = 0, {}
    for qid, q in CHECKLIST.items():
        r = client.system_one(state=policy, questions={qid: q})
        single_tokens += r.usage.input_tokens
        single_answers[qid] = r.nouls[qid].noul
    single_seconds = time.perf_counter() - t0

    # all eight in one call
    t0 = time.perf_counter()
    batched = client.system_one(state=policy, questions=CHECKLIST)
    batch_seconds = time.perf_counter() - t0
    batch_tokens = batched.usage.input_tokens

print(f"{'':<16}{'seconds':>10}{'input tokens':>14}{'cost':>14}")
print(f"{'8 calls':<16}{single_seconds:>10.2f}{single_tokens:>14,}{single_tokens / 1e6 * PRICE_PER_MTOK:>14.8f}")
print(f"{'1 call':<16}{batch_seconds:>10.2f}{batch_tokens:>14,}{batch_tokens / 1e6 * PRICE_PER_MTOK:>14.8f}")
print(f"{'ratio':<16}{single_seconds / batch_seconds:>9.1f}x{single_tokens / batch_tokens:>13.1f}x")

print("\nanswers agree per question:")
for qid in CHECKLIST:
    print(f"  {qid:<24} single={single_answers[qid]:.2f}  batched={batched.nouls[qid].noul:.2f}")

                   seconds  input tokens          cost
8 calls               0.83         2,840    0.00011928
1 call                0.12           467    0.00001961
ratio                 6.8x          6.1x

answers agree per question:
  retention_defined        single=0.99  batched=0.99
  backup_lifecycle         single=0.98  batched=0.98
  data_export              single=0.99  batched=0.99
  third_party_sharing      single=0.98  batched=0.98
  sharing_purpose_limited  single=0.94  batched=0.95
  change_notification      single=0.96  batched=0.96
  deletion_on_closure      single=0.91  batched=0.90
  mentions_encryption      single=0.01  batched=0.01


There is **no limit on the number of questions per request**; the bound is context size
(each request must fit the per-request context, and the state plus any one question must fit
the per-question context; notebook 05 has the numbers). If the questions do not all fit,
spread them over several requests with the same state.

When is a second request genuinely required? Only when your code cannot *build* the next
question until it has an answer: fetching different data for the state, or choosing the next
question's options from the previous answer (hierarchical classification walks a taxonomy
level by level this way). Otherwise: one request, speculative questions, code ignores the rest.

## Pattern 2: Confidence-gated routing

Covered in depth in notebook 03; the pattern in one sentence: **the answer tells you what,
confidence tells you whether to act, and riskier actions get higher bars.** The banking
router in notebook 03 is the canonical example. Below we reuse the idea inside a bigger
system, so we are not repeating it here.

## Pattern 3: Composite scoring

To rank items by a judgment that depends on several things, resist the one-big-question
urge ("rate this candidate 1 to 10"). Score each dimension separately, normalize, and
combine with weights that live in your code. When the ranking does not match what your team
would decide, you change a coefficient under code review, not a prompt.

Normalization matters: a 5-level scale returns 0 to 4 and a 3-level scale 0 to 2, so divide
each score by `len(criteria) - 1` before weighting.

In [6]:
import pandas as pd
from typesafe_sdk import Score, TypeSafeClient

DIMENSIONS = {
    "python_depth": Score(
        instructions="How deep is the candidate's Python experience?",
        criteria=[
            "No Python mentioned",
            "Occasional scripts or coursework",
            "Regular professional use",
            "Multiple years of daily professional use",
            "Expert: owns large production Python systems",
        ],
    ),
    "team_leadership": Score(
        instructions="How much team leadership experience does the resume show?",
        criteria=[
            "None mentioned",
            "Mentored individuals or led small projects",
            "Led a team formally",
            "Led multiple teams or an organization",
        ],
    ),
    "system_design": Score(
        instructions="How much system design experience does the resume show?",
        criteria=[
            "None mentioned",
            "Contributed to designs owned by others",
            "Designed significant components independently",
            "Owned architecture for large production systems",
        ],
    ),
}

resumes = {
    "Asha": "Eight years of Python, designed and operates a 40-service event pipeline, mentors two juniors.",
    "Ben": "Engineering manager for 5 years across two teams; writes occasional Python tooling; led the redesign of our billing architecture with the staff engineer.",
    "Chloe": "Fresh graduate, strong coursework in Python and algorithms, internship building a REST API.",
}

WEIGHTS = {
    "senior_ic": {"python_depth": 0.45, "team_leadership": 0.10, "system_design": 0.45},
    "eng_manager": {"python_depth": 0.15, "team_leadership": 0.55, "system_design": 0.30},
}

rows = []
with TypeSafeClient() as client:
    for name, resume in resumes.items():
        r = client.system_one(state=resume, questions=DIMENSIONS)
        normalized = {
            qid: r.scores[qid].score / (len(DIMENSIONS[qid].criteria) - 1)
            for qid in DIMENSIONS
        }
        row = {"candidate": name, **{k: round(v, 2) for k, v in normalized.items()}}
        for role, w in WEIGHTS.items():
            row[role] = round(sum(w[q] * normalized[q] for q in DIMENSIONS), 3)
        rows.append(row)

pd.DataFrame(rows).sort_values("senior_ic", ascending=False)

,candidate,python_depth,team_leadership,system_design,senior_ic,eng_manager
0,Asha,0.99,0.33,0.95,0.909,0.618
1,Ben,0.25,1.00,0.69,0.525,0.794
2,Chloe,0.33,0.00,0.15,0.217,0.095


Same three resumes, two rankings, and the difference between them is three numbers in
`WEIGHTS` that anyone on the team can read, review, and change. The model's judgments
(the per-dimension scores) stay untouched. That separation is the pattern.

## Pattern 4: Intent routing

Not every request deserves the same handler. Some need a database lookup, some need an
expensive specialist LLM, some need a human. Jev sits in front as a fast, cheap classifier,
and the expensive resources are invoked only for requests that need them.

In [7]:
from typesafe_sdk import Choice, Score, TypeSafeClient

ROUTER = {
    "intent": Choice(
        instructions="What does the customer want?",
        criteria={
            "order_status": "Where is my order, tracking, delivery dates",
            "product_question": "How a product works, specs, compatibility",
            "return_exchange": "Returning or exchanging an item",
            "complaint": "Expressing dissatisfaction with service or product",
        },
    ),
    "complexity": Score(
        instructions="How complex is this request to resolve?",
        criteria=[
            "Single lookup or standard answer",
            "Needs some back-and-forth or judgment",
            "Multi-issue, legal threat, or needs escalation",
        ],
    ),
}


def route(message: str, answers) -> str:
    intent, complexity = answers["intent"], answers["complexity"]
    if intent.confidence < 0.5:
        return "human agent (unclear intent)"
    if intent.choice == "order_status":
        return "deterministic code: order lookup, no LLM involved"
    if intent.choice == "product_question":
        return "product specialist LLM"
    if intent.choice == "return_exchange":
        return "returns specialist LLM"
    # complaint: use the complexity score to decide between an LLM and a person
    if complexity.score > 1 or complexity.confidence < 0.5:
        return "human agent (complex complaint)"
    return "complaint resolution LLM"


messages = [
    "where is order A-1043?",
    "does the X200 work with USB-C docks?",
    "this is the third defective unit, I am contacting my lawyer",
    "slightly annoyed the color is different from the photos, can I swap it?",
    "???",
]

with TypeSafeClient() as client:
    for m in messages:
        r = client.system_one(state=m, questions=ROUTER)
        print(f"{m!r:<64} -> {route(m, r.answers)}")

'where is order A-1043?'                                         -> deterministic code: order lookup, no LLM involved
'does the X200 work with USB-C docks?'                           -> product specialist LLM
'this is the third defective unit, I am contacting my lawyer'    -> human agent (complex complaint)


'slightly annoyed the color is different from the photos, can I swap it?' -> returns specialist LLM
'???'                                                            -> product specialist LLM


The routing call costs a few hundred input tokens (fractions of a cent per thousand
messages) and saves an expensive LLM invocation for every message that deterministic code or
a human should handle anyway.

## Exercises

1. Fan-out: add two speculative questions to `TRIAGE` (language of the ticket, mentions a
   named competitor) and extend the routing code to use one of them.
2. Batching: rerun the economics cell with the checklist against a state 10x longer. Does the
   token ratio grow or shrink? Why?
3. Composite: add a `communication` dimension to the resume screen, re-weight, and check how
   the two rankings move.
4. Routing: add an `angry` Noul to `ROUTER` and route any message with `noul > 0.9` to a
   human regardless of intent.